# Project 1: Differentiated Demand for Cars

## Theoretical Question 3 — Testing the Perloff-Salop Model Empirically

Perloff-Salop (1985) predicts a symmetric equilibrium price:

$$
p^* = c + \frac{1}{n(n-1)\int F(\varepsilon)^{n-2} f(\varepsilon)^2\,d\varepsilon}
$$

Price therefore depends only on marginal cost $c$ and the intensity of competition $n$, together with the shape of the match-value distribution $F$, which is treated as fixed/structural.

Crucially, price should **not** depend on a product's own observed characteristics such as weight or fuel efficiency. In the baseline Perloff-Salop model, firms are otherwise symmetric and differentiation works through the i.i.d. match value rather than through systematic vertical quality that would affect equilibrium pricing.

### Empirical test

We regress price on:

- a proxy for marginal cost,
- the number of competing firms $n$,
- taxes,
- weight (`we`), and
- fuel efficiency (`li`).

We then conduct a joint F-test of:

$$
H_0: \beta_{we}=0 \quad \text{and} \quad \beta_{li}=0.
$$

Rejecting $H_0$ provides evidence against the baseline Perloff-Salop pricing restriction.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


In [ ]:
# Load data
df = pd.read_csv("cars.csv")

# Define a market as a year-country combination
df["market"] = df["ye"].astype(str) + "_" + df["ma"].astype(str)

# Number of distinct competing firms in each market
# This is used as a proxy for n in the Perloff-Salop model.
df["n_firms"] = df.groupby("market")["frm"].transform("nunique")

df.head()


In [ ]:
# Cost proxy: producer price index of the exporting country (avppr)
# Remove observations for which logs cannot be constructed.
df = df[df["avppr"] > 0].dropna(
    subset=["avppr", "eurpr", "tax", "n_firms", "we", "li"]
).copy()

df = df[df["eurpr"] > 0].copy()

# Log transformations
df["log_avppr"] = np.log(df["avppr"])
df["log_eurpr"] = np.log(df["eurpr"])

df[["log_eurpr", "log_avppr", "tax", "n_firms", "we", "li"]].describe()


In [ ]:
# Estimate the pricing regression
X = sm.add_constant(df[["log_avppr", "tax", "n_firms", "we", "li"]])
y = df["log_eurpr"]

# Standard errors are clustered at the market level.
model = sm.OLS(y, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["market"]}
)

print(model.summary())


In [ ]:
# Joint test of the Perloff-Salop restriction on observed characteristics
# H0: coefficient on weight = 0 AND coefficient on fuel efficiency = 0
ftest = model.f_test("we = 0, li = 0")

print("Joint test of H0: coef(we) = 0 and coef(li) = 0")
print(ftest)


### Interpretation

- If the joint test **fails to reject** $H_0$, the data do not provide evidence that weight and fuel efficiency systematically affect prices after controlling for the cost proxy, taxes, and the number of firms. This is consistent with the tested pricing restriction of the baseline Perloff-Salop model.
- If the joint test **rejects** $H_0$, observed product characteristics have explanatory power for prices. This is evidence against the baseline Perloff-Salop pricing restriction.

The test should be interpreted as a test of this particular empirical implication of the model rather than as a test of every feature of the Perloff-Salop framework.